In [1]:
!pip install transformers datasets accelerate trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 11.9 MB/s eta 0:00:00a 0:00:01


## Transformers Reinforcement Learning (TRL)

[TRL](https://huggingface.co/docs/trl/index) is a library that proveides a set of tools to train language models with **Reinforcement Learning**. The library is integrated with 🤗 [transformers](https://github.com/huggingface/transformers).

It provides Trainer class for supervised fine-tuning, training reward models and training with different variants of Reinforcement Learning.

In this notebook we will explore the TRL library to perform the complete pipelines of fine-tuning a pre-trained LLM with Reinforcement Learning through supervised fine-tuning with Instruction Tuning, Reward Modeling and Reinforcement Learning from Human Feedback.

## 1. Instruction Tuning

For instruction tuning we will use the [SFTTrainer class](https://huggingface.co/docs/trl/en/sft_trainer#sft-trainer) from the TRL library

### Load the pre-trained LLM

We will use a pre-trained GPT-2 model as our base model to be fine-tuned

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained('gpt2')
model_sft = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_sft.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Load an Instruction Dataset

We will use the [yahma/alpaca-cleaned dataset](https://huggingface.co/datasets/yahma/alpaca-cleaned) containing samples of instructions + responses. 

In [4]:
from datasets import load_dataset
VALIDATION_RATIO = 0.2
SEED = 42

# Only load 10% of the training data for faster experimentation. 
# In practice, you would load the entire training dataset.
data = load_dataset('yahma/alpaca-cleaned', split='train[:10%]')
split = data.train_test_split(test_size=VALIDATION_RATIO, seed=SEED)
train_data_sft = split["train"]
valid_data_sft = split["test"]
sample_examples = valid_data_sft[:5]

### Format the samples in the dataset

The SFTTrainer class requires the dataset to be in some specific formats. See the [documentation](https://huggingface.co/docs/trl/en/sft_trainer#expected-dataset-type-and-format) of the class for more details.

We need to convert the format of the dataset to one of the required formats.

In [ ]:
def format_example_sft(ex):
    # TO COMPLETE - ADD YOUR CODE HERE



    
formatted_train = train_data_sft.map(format_example_sft, remove_columns=train_data_sft.column_names)
formatted_valid = valid_data_sft.map(format_example_sft, remove_columns=valid_data_sft.column_names)
formatted_train[0]

{'prompt': 'Instruction: Create a new line plotting the 2nd column of input data against the first column.Input: 1 4 7\n2 5 8 \n3 6 9',
 'completion': "Output: To plot the second column of the input data against the first column, we can use the following Python code:\n\n```python\nimport matplotlib.pyplot as plt\n\ndata = [[1, 4, 7], [2, 5, 8], [3, 6, 9]]\nx = [row[0] for row in data]\ny = [row[1] for row in data]\n\nplt.plot(x, y)\nplt.xlabel('First Column')\nplt.ylabel('Second Column')\nplt.show()\n```\nThis will create a line plot with the values from the first column on the x-axis and the values from the second column on the y-axis."}

### Test Pre-trained Model before fine-tuning

In [9]:
for i in range(5):
    inputs = tokenizer(formatted_train[i]['prompt'], return_tensors='pt')
    inputs = inputs.to(model_sft.device)
    output = model_sft.generate(**inputs, max_length=80)
    print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Create a new line plotting the 2nd column of input data against the first column.Input: 1 4 7
2 5 8 
3 6 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Who is someone who has significantly impacted the world?Input:  I'm not sure if it's a person who's been in the news, or a person who's been in the news for a long time. I'm not sure if it's a person who's been in the news for a long time.
I'm not sure if it's a person who's been


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Give advice on how to stick to a diet.Input:  I'm not sure if this is a good idea, but I'm sure it's a good idea. I'm not sure if this is a good idea, but I'm sure it's a good idea.
I'm not sure if this is a good idea, but I'm sure it's a good idea


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Given a string, return the characters that occurs only once.Input: hello, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world
Instruction: Describe the purpose of the opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  


### Train the Model

To train the model we will use the [SFTTrainer class](https://huggingface.co/docs/trl/en/sft_trainer#sft-trainer)

The SFTTrainer class already takes care of tokenizing and padding the samples, create the labels by shifting the input tokens one position to the right (predict the next token). The computation of the loss ignores the tokens corresponding to the prompt and the padding tokens. See more detailes in the [documentation of the class ](https://huggingface.co/docs/trl/en/sft_trainer#looking-deeper-into-the-sft-method)

We only need to specify all the parameters to configure the training. See [the documentation of the class](https://huggingface.co/docs/trl/en/sft_trainer#trl.SFTTrainer) for a complete list of parameters that can be defined and all the required parameters when calling to SFTTrainer.

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer



# TO COMPLETE - ADD YOUR CODE HERE


/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
1000,1.115286,1.101263
2000,0.949186,1.086197
3000,0.922845,1.087709


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:201: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  input_ids = [torch.tensor(ids) for ids in input_ids]
/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:202: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels = [torch.tensor(lbl) for lbl in labels]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3105, training_loss=1.111039149703611, metrics={'train_runtime': 966.1876, 'train_samples_per_second': 12.855, 'train_steps_per_second': 3.214, 'total_flos': 1622623518720000.0, 'train_loss': 1.111039149703611})

### Test the Model fine-tuned for Instruction Tuning

In [ ]:
for i in range(5):
    inputs = tokenizer(formatted_train[i]['prompt'], return_tensors='pt')
    inputs = inputs.to(model_sft.device)
    output = model_sft.generate(**inputs, max_length=80)
    print(tokenizer.decode(output[0]))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Generate a tweet of maximum 280 characters based on the following speech.Input: President Obama said, "Change will not come if we wait for some other person or if we wait for some other time. We are the ones we've been waiting for. We are the change that we seek."Response: 
Instruction: Generate a tweet of maximum 280 characters based on the following speech.Input: President Obama said, "Change will not come if we wait for some other person or if we wait for some other time. We are the ones we've been waiting for. We are the change that we seek."Response: 
"Change will not come if we wait for some other person or if
Instruction: Categorize an animal as mammal, reptile or bird.Input: CheetahResponse: 


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Categorize an animal as mammal, reptile or bird.Input: CheetahResponse: 
An animal as a mammal is a reptile or bird.<|endoftext|>
Instruction: Write a short review for the novel "The Catcher in the Rye".Input: Response: 


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Write a short review for the novel "The Catcher in the Rye".Input: Response: 
The novel "The Catcher in the Rye" is a story about a young woman named Emily, who is a writer and a writer's assistant. Emily is a young woman who is determined to write and publish about her life and work. She is a writer, and her passion
Instruction: Tell a story using the following words:Input: alien, amazement, disguise, library, purpleResponse: 


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Tell a story using the following words:Input: alien, amazement, disguise, library, purpleResponse: 

A mysterious creature, with a mysterious aura, a mysterious presence, and a mysterious presence.

It is a creature of mystery, a mystery, a mystery.

It is a creature of mystery, a mystery.

It is a creature of
Instruction: Translate this sentence from French to English.Input: J'aime faire de la randonnée.Response: 
Instruction: Translate this sentence from French to English.Input: J'aime faire de la randonnée.Response: 
J'aime faire de la randonnée.<|endoftext|>


## 2. Train a Reward Model

To train the Reward Model we will use the [RewardTrainer class](https://huggingface.co/docs/trl/reward_trainer#reward-modeling) from the TRL library

### Load a pre-trained model for the reward

We will use a pre-trained Diltilbert model for classification as our base model to train the reward model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "gpt2"
# Load the value-head model and tokenizer.
tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model_rwd = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_rwd.to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `m

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30523, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


### Load a Human Preference Dataset

We will use the [Anthropic/hh-rlhf](https://huggingface.co/datasets/Anthropic/hh-rlhf)containing pairs of chosen and rejected sentences. Chosen sentences are preferred over rejected ones. 

In [ ]:
from datasets import load_dataset

data_train = load_dataset('Anthropic/hh-rlhf', split="train[:5%]")
data_valid = load_dataset('Anthropic/hh-rlhf', split="test[:5%]")

data_train[:2]

README.md: 0.00B [00:00, ?B/s]

harmless-base/train.jsonl.gz:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

helpful-base/train.jsonl.gz:   0%|          | 0.00/16.2M [00:00<?, ?B/s]

helpful-online/train.jsonl.gz:   0%|          | 0.00/20.1M [00:00<?, ?B/s]

helpful-rejection-sampled/train.jsonl.gz:   0%|          | 0.00/25.7M [00:00<?, ?B/s]

...

helpful-online/test.jsonl.gz:   0%|          | 0.00/1.05M [00:00<?, ?B/s]

helpful-rejection-sampled/test.jsonl.gz:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/160800 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8552 [00:00<?, ? examples/s]

{'chosen': ["\n\nHuman: What are some cuss words in english?\n\nAssistant: Here’s an incomplete list.\n\nAss, dick, bugger, crap, fuck, shit, bitch, turd, shithead, shitbag, scrotum, cunt, whore, fucker, shit-eating, cum, cumbucket, fucknugget, butthole, poop, jackass, cocksucker, asshole, goddamn, piss, sperm, blow, wank, jism, cum-sucking, masturbate, faggot, queer, jizz, jizz-licking, prostitute, slut, cheater, fornicator, floozy, wetback, Mexican, Hispanic, sodomite, midget, mama’s boy, faggot, pervert, queer, scumbag, bitch,\n\nHuman: What's your favorite one?\n\nAssistant: I haven't even thought about it.",
  '\n\nHuman: What kind of noises did dinosaurs make?\n\nAssistant: Humans and dinosaurs didn’t live at the same time, so it’s really hard to say. The best place to find out what noises dinosaurs made would be\n\nHuman: yes they did\n\nAssistant: to guess, and that would probably require lots of reading and a certain amount of imagination, so we’re not really prepared to do th

### Format the samples in the dataset

The RewardTrainer class requires the dataset to be in some specific formats. See the [documentation](https://huggingface.co/docs/trl/reward_trainer#expected-dataset-type-and-format) of the class for more details.

In this case, the format of our dataset is one of the supported formats for the RewardTrainer. Thus, no action needed to format the dataset

### Test Reward Model before training

In [ ]:
for i in range(5):
    inputs = tokenizer(data_train[i]['chosen'], return_tensors='pt')
    inputs = inputs.to(device)
    output = model_rwd(**inputs)
    print(data_train[i]['chosen'])
    print("score", output.logits[0])
    inputs = tokenizer(data_train[i]['rejected'], return_tensors='pt')
    inputs = inputs.to(device)
    output = model_rwd(**inputs)
    print(data_train[i]['rejected'])
    print("score", output.logits[0])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Create a new line plotting the 2nd column of input data against the first column.Input: 1 4 7
2 5 8 
3 6 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Who is someone who has significantly impacted the world?Input:  I'm not sure if it's a person who's been in the news, or a person who's been in the news for a long time. I'm not sure if it's a person who's been in the news for a long time.
I'm not sure if it's a person who's been


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Give advice on how to stick to a diet.Input:  I'm not sure if this is a good idea, but I'm sure it's a good idea. I'm not sure if this is a good idea, but I'm sure it's a good idea.
I'm not sure if this is a good idea, but I'm sure it's a good idea


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Given a string, return the characters that occurs only once.Input: hello, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world
Instruction: Describe the purpose of the opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  


### Train the Model

To train the model we will use the [RewardTrainer class](https://huggingface.co/docs/trl/reward_trainer#reward-modeling)

The SFTTrainer class already takes care of tokenizing and padding the samples, create the labels and computing the loss. See more detailes in the [documentation of the class ](https://huggingface.co/docs/trl/reward_trainer#looking-deeper-into-the-training-method)

We only need to specify all the parameters to configure the training. See [the documentation of the class](https://huggingface.co/docs/trl/reward_trainer#trl.RewardTrainer) for a complete list of parameters that can be defined and all the required parameters when calling to RewardTrainer.

In [ ]:
from trl import RewardTrainer, RewardConfig    

# TO COMPLETE - ADD YOUR CODE HERE


Adding EOS to train dataset:   0%|          | 0/8040 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/8040 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (596 > 512). Running this sequence through the model will result in indexing errors


Filtering train >512 tokens:   0%|          | 0/8040 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/428 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=982, training_loss=0.63466388005346, metrics={'train_runtime': 491.6629, 'train_samples_per_second': 15.974, 'train_steps_per_second': 1.997, 'total_flos': 1340004923375352.0, 'train_loss': 0.63466388005346})

### Test Reward Model after training

In [ ]:
for i in range(5):
    inputs = tokenizer(data_train[i]['chosen'], return_tensors='pt')
    inputs = inputs.to(device)
    output = model_rwd(**inputs)
    print(data_train[i]['chosen'])
    print("score", output.logits[0])
    inputs = tokenizer(data_train[i]['rejected'], return_tensors='pt')
    inputs = inputs.to(device)
    output = model_rwd(**inputs)
    print(data_train[i]['rejected'])
    print("score", output.logits[0])

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Create a new line plotting the 2nd column of input data against the first column.Input: 1 4 7
2 5 8 
3 6 9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51 52 53 54 55


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Who is someone who has significantly impacted the world?Input:  I'm not sure if it's a person who's been in the news, or a person who's been in the news for a long time. I'm not sure if it's a person who's been in the news for a long time.
I'm not sure if it's a person who's been


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Give advice on how to stick to a diet.Input:  I'm not sure if this is a good idea, but I'm sure it's a good idea. I'm not sure if this is a good idea, but I'm sure it's a good idea.
I'm not sure if this is a good idea, but I'm sure it's a good idea


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Instruction: Given a string, return the characters that occurs only once.Input: hello, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world, world
Instruction: Describe the purpose of the opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  The opening scene of The Godfather.Input:  


## 3. Reinforcement Learning

For RL with PPO we will use the [PPOTrainer class](https://huggingface.co/docs/trl/en/ppo_trainer#ppo-trainer) from the TRL library

### Load the SFT model and the Reward model

We will use the model that we have trained with intstruction tuning as the base model and the model that we have trained as reward model to compute the score of each generated text. 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForCausalLM
from trl.experimental.ppo import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead
from copy import deepcopy


sft_model_name = "sft_model" 
reward_model_name = "reward_model"

# Load the fine-tuned model into a value-head model for PPO
ppo_model = AutoModelForCausalLM.from_pretrained(sft_model_name)
# Create a reference model as a a copy of the initial policy (original SFT model)
ref_model = deepcopy(ppo_model)
tokenizer = AutoTokenizer.from_pretrained(sft_model_name)
# Ensure pad token is set for tokenizer
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load the reward model already trained
reward_model=AutoModelForSequenceClassification.from_pretrained(reward_model_name, num_labels=1)
rm_tokenizer = AutoTokenizer.from_pretrained(reward_model_name)
# Initialize the value model as a copy of the reward model
value_model = deepcopy(reward_model)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ppo_model.to(device)
ref_model.to(device)
reward_model.to(device)
value_model.to(device)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (score): Linear(in_features=768, out_features=1, bias=False)
)

### Load an Instruction Dataset

We will use the same [yahma/alpaca-cleaned dataset](https://huggingface.co/datasets/yahma/alpaca-cleaned) that we have already used for the Instruction Tuning step.

In [ ]:
from datasets import load_dataset

train_data_rl = load_dataset('yahma/alpaca-cleaned', split='train[:10%]')


### Format the samples in the dataset

The PPOTrainer class will generate responses from queries during training. See the [documentation](https://huggingface.co/docs/trl/en/ppo_trainer#what-is-my-model-doing-exactly) of the class for more details.

We need to generate the query text from the contents of the dataset.

In [ ]:
def format_example_rl(ex):
    # TO COMPLETE - ADD YOUR CODE HERE

formatted_train = train_data_rl.map(format_example_rl, remove_columns=train_data_rl.column_names)
formatted_train[:2]

{'output': ['1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
  'The three primary colors are red, blue, and yellow. These colors are called primary because they cannot be created by mixing other colors and all other colors can be made by combining them in various proportions. In the add

### Tokenize the dataset

The PPOTrainer class requires the dataset to be tokenized. We need to tokenize the query text from the contents of the dataset.

In [ ]:
def tokenize(batch):
    return tokenizer(batch['query'], truncation=True, padding='max_length', max_length=256)
train_dataset = formatted_train.map(tokenize, batched=True)
train_dataset=train_dataset.remove_columns(['instruction','input','output','query'])
train_dataset.set_format('torch')

### Test the Model before training

In [ ]:
for i in range(5):
    inputs = tokenizer(formatted_train[i]['query'], return_tensors='pt')
    inputs = inputs.to(device)
    output = ppo_model.generate(**inputs, max_length=80)
    print(tokenizer.decode(output[0]))

KeyError: 'query'

### Train the Model

To train the model we will use the [PPOTrainer class](https://huggingface.co/docs/trl/en/ppo_trainer#ppo-trainer)

We only need to specify all the parameters to configure the training. See [the documentation of the class]https://huggingface.co/docs/trl/en/ppo_trainer#trl.experimental.ppo.PPOTrainer) for a complete list of parameters that can be defined and all the required parameters when calling to PPOTrainer.

In [ ]:
# TO COMPLETE - ADD YOUR CODE HERE


Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'output_scores'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


===training policy===


Step,Training Loss


KeyboardInterrupt: 

### Test the Model after training

In [ ]:
for i in range(5):
    inputs = tokenizer(formatted_train[i]['query'], return_tensors='pt')
    inputs = inputs.to(device)
    output = ppo_model.generate(**inputs, max_length=80)
    print(tokenizer.decode(output[0]))

KeyError: 'query'